# 07 - Dimension Tables

## Objective

The objective of this notebook is to create and validate the dimension tables required for the analytical data model.

Dimension tables contain descriptive information about business entities and are used to provide context for analysis.

## Source Tables

The dimension tables are created from the cleaned Silver tables:

- `clean_customers`
- `clean_orders`
- `clean_products`
- `clean_order_payments`
- Date information derived from order dates

## Dimension Tables Created

The following dimension tables are created:

- `dim_customer` — Customer information and location details
- `dim_product` — Product attributes and category information
- `dim_order` — Order-level information and order status
- `dim_payment` — Payment-related information
- `dim_date` — Calendar information used for time-based analysis

## Data Modeling Approach

The dimension tables are designed to support analytical queries and provide descriptive attributes for the fact tables.

The analytical model follows a dimensional modeling approach where dimension tables are connected to the central fact tables using business keys such as:

- `customer_id`
- `product_id`
- `order_id`
- `date_key`

## Validation

After creating the dimension tables, the following checks are performed:

- Row counts
- Unique key counts
- NULL primary keys
- Duplicate keys
- Date coverage
- Data consistency

## Validation Results

The dimension tables were successfully created and validated.

Key results:

- `dim_customer`: 99,441 customers
- `dim_product`: 32,951 products
- `dim_order`: 99,441 orders
- `dim_payment`: 5 payment types
- `dim_date`: 774 unique dates

## Outcome

The dimension layer is ready to support the analytical fact tables and downstream Gold business tables.

In [0]:
%sql

SELECT
    MIN(order_purchase_timestamp) AS min_order_date,
    MAX(order_purchase_timestamp) AS max_order_date
FROM clean_orders;

####Create dim_date

In [0]:
%sql

CREATE OR REPLACE TABLE dim_date AS

SELECT
    date AS date_key,
    YEAR(date) AS year,
    QUARTER(date) AS quarter,
    MONTH(date) AS month,
    DATE_FORMAT(date, 'MMMM') AS month_name,
    DAY(date) AS day,
    DAYOFWEEK(date) AS day_of_week,
    DATE_FORMAT(date, 'EEEE') AS day_name
FROM (
    SELECT
        EXPLODE(
            SEQUENCE(
                TO_DATE('2016-09-04'),
                TO_DATE('2018-10-17'),
                INTERVAL 1 DAY
            )
        ) AS date
    );

In [0]:
%sql

SELECT *
FROM dim_date
ORDER BY date_key
LIMIT 10;

In [0]:
%sql

SELECT
    MIN(date_key) AS min_date,
    MAX(date_key) AS max_date,
    COUNT(*) AS total_dates
FROM dim_date;

####dim_payment

In [0]:
%sql

SELECT
    payment_type,
    COUNT(*) AS transaction_count,
    ROUND(SUM(payment_value), 2) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS average_payment_value
FROM clean_order_payments
GROUP BY payment_type
ORDER BY transaction_count DESC;

In [0]:
%sql

CREATE OR REPLACE TABLE dim_payment AS

SELECT DISTINCT
    payment_type
FROM clean_order_payments
WHERE payment_type IS NOT NULL;

In [0]:
%sql

SELECT *
FROM dim_payment
ORDER BY payment_type;

####dim_product

In [0]:
%sql

CREATE OR REPLACE TABLE dim_product AS
SELECT
    product_id,
    product_category_name,
    product_name_lenght,
    product_description_lenght,
    product_photos_qty,
    product_weight_g,
    product_length_cm,
    product_height_cm,
    product_width_cm
FROM clean_products

In [0]:
%sql
SHOW TABLES;

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
SELECT COUNT(*) AS product_count
FROM clean_products;

In [0]:
%sql
SELECT COUNT(*) AS dimension_product_count
FROM dim_product;

In [0]:
%sql
SELECT COUNT(*) AS clean_customer_count
FROM clean_customers;

SELECT COUNT(*) AS dimension_customer_count
FROM dim_customer;

In [0]:
%sql
SELECT COUNT(*) AS clean_order_count
FROM clean_orders;

SELECT COUNT(*) AS dimension_order_count
FROM dim_order;

In [0]:
%sql
SELECT COUNT(*) AS clean_payment_count
FROM clean_order_payments;

SELECT COUNT(*) AS dimension_payment_count
FROM dim_payment;

####validate dim_date

In [0]:
%sql
SELECT
    MIN(date_key) AS min_date,
    MAX(date_key) AS max_date,
    COUNT(*) AS total_dates,
    COUNT(DISTINCT date_key) AS unique_dates
FROM dim_date;